In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

pozisyonlar = [
    "İş Analisti", "Veri Analisti", "Yazılım Geliştirici", 
    "Proje Yöneticisi", "Veri Bilimci", "İnsan Kaynakları Uzmanı",
    "Dijital Pazarlama Uzmanı", "Finans Uzmanı", "Sistem Mühendisi", "İş Geliştirme Uzmanı"
]

def ilanlari_topla():
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options)
    tum_veriler = []

    try:
        for pozisyon in pozisyonlar:
            print(f"--- {pozisyon} için ilanlar toplanıyor... ---")
            
            url = f"https://www.kariyer.net/is-ilanlari?kw={pozisyon}"
            driver.get(url)
            
            if pozisyon == pozisyonlar[0]:
                print("İlk aramada botu manuel geçmeniz gerekebilir. 15 saniye bekleniyor...")
                time.sleep(15)
            else:
                time.sleep(5) 

            try:
                wait = WebDriverWait(driver, 15)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-test="ad-card"]')))

                ilan_kartlari = driver.find_elements(By.CSS_SELECTOR, '[data-test="ad-card"]')
                
                for ilan in ilan_kartlari:
                    try:
                        baslik = ilan.find_element(By.CLASS_NAME, "k-ad-card-title").text
                        sirket = ilan.find_element(By.CSS_SELECTOR, '[data-test="subtitle"]').text
                        link = ilan.find_element(By.TAG_NAME, "a").get_attribute("href")
                        
                        tum_veriler.append({
                            "Aranan_Pozisyon": pozisyon,
                            "İlan_Başlığı": baslik,
                            "Şirket": sirket,
                            "Link": link
                        })
                    except:
                        continue
                print(f"{pozisyon} için {len(ilan_kartlari)} adet ilan eklendi.")

            except Exception as e:
                print(f"{pozisyon} aranırken bir hata oluştu veya ilan bulunamadı.")

        df = pd.DataFrame(tum_veriler)
        return df

    finally:
        driver.quit()

sonuc_df = ilanlari_topla()

if not sonuc_df.empty:
    print("\nToplam Toplanan İlan Sayısı:", len(sonuc_df))
    sonuc_df.to_csv("kariyer_toplu_ilanlar.csv", index=False, encoding="utf-8-sig")
    print("Veriler 'kariyer_toplu_ilanlar.csv' dosyasına kaydedildi.")
else:
    print("Hiç veri toplanamadı.")

In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

pozisyonlar = [
    "İş Analisti", "Veri Analisti", "Yazılım Geliştirici", 
    "Proje Yöneticisi", "Veri Bilimci", "İnsan Kaynakları Uzmanı",
    "Dijital Pazarlama Uzmanı", "Finans Uzmanı", "Sistem Mühendisi", "İş Geliştirme Uzmanı"
]

SAYFA_SAYISI = 20 

def ilanlari_topla():
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options)
    tum_veriler = []

    try:
        for pozisyon in pozisyonlar:
            for sayfa in range(1, SAYFA_SAYISI + 1):
                print(f"--- {pozisyon} | Sayfa: {sayfa} toplanıyor... ---")
                
                url = f"https://www.kariyer.net/is-ilanlari?kw={pozisyon}&cp={sayfa}"
                driver.get(url)
                
                # Bot engeli kontrolü
                if pozisyon == pozisyonlar[0] and sayfa == 1:
                    print("Lütfen botu manuel geçin (Gerekirse)... 15 saniye bekleniyor.")
                    time.sleep(15)
                else:
                    time.sleep(4) 

                try:
                    wait = WebDriverWait(driver, 15)
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-test="ad-card"]')))
   
                    ilan_kartlari = driver.find_elements(By.CSS_SELECTOR, '[data-test="ad-card"]')
                    
                    if not ilan_kartlari:
                        print(f"{pozisyon} için bu sayfada ilan bulunamadı, sonraki pozisyona geçiliyor.")
                        break 

                    for ilan in ilan_kartlari:
                        try:
                            baslik = ilan.find_element(By.CLASS_NAME, "k-ad-card-title").text
                            sirket = ilan.find_element(By.CSS_SELECTOR, '[data-test="subtitle"]').text
                            link = ilan.find_element(By.TAG_NAME, "a").get_attribute("href")
                            
                            tum_veriler.append({
                                "Aranan_Pozisyon": pozisyon,
                                "Sayfa_No": sayfa,
                                "İlan_Başlığı": baslik,
                                "Şirket": sirket,
                                "Link": link
                            })
                        except:
                            continue
                    
                except Exception as e:
                    print(f"{pozisyon} - Sayfa {sayfa} yüklenirken hata oluştu veya ilan bitti.")
                    break 
        df = pd.DataFrame(tum_veriler)
        return df

    finally:
        driver.quit()

sonuc_df = ilanlari_topla()

if not sonuc_df.empty:
    print(f"\nİşlem Tamamlandı! Toplam {len(sonuc_df)} ilan toplandı.")
    sonuc_df.to_csv("kariyer_detayli_liste.csv", index=False, encoding="utf-8-sig")
else:
    print("Veri toplanamadı. Lütfen bot engelini ve internet bağlantınızı kontrol edin.")